[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zjelveh/zjelveh.github.io/blob/master/files/ccjs418e_fall2026/5_pandas_filtering_lecture.ipynb)

**IMPORTANT: Save your own copy.**
1. Click File > Save a copy in Drive.
2. Rename it.
3. Work in your copy, not the original.

---

# 5. Filtering DataFrames: Building Comparison Groups
## CCJS 418E: Coding for Criminology

Today we will use conditions to select the records needed for a comparison. We will combine conditions, select several categories with `.isin()`, calculate ratios and percentages, create new columns, and complete an analysis from question to answer.

Our main question is:

> In 2019, which state had the higher reported property-crime rate, Maryland or Virginia? Did that state also have a larger percentage of property crime represented by motor vehicle theft?


## Part 1: From a Comparison to Rows in a Table

In the body-camera reading, the researchers compared officers assigned to wear body cameras with officers assigned not to wear them. If those officers were rows in a DataFrame, a column could record each officer's assignment. A condition on that column would identify the two groups.

Filtering is how we select the rows that meet a condition. The condition comes from the question and comparison we have chosen.

Today we will practice with the same state-crime table from the previous class. One row represents **one state in one year**, and every crime measure is a rate per 100,000 people.


In [ ]:
import pandas as pd

data_url = "https://raw.githubusercontent.com/zjelveh/zjelveh.github.io/refs/heads/master/files/ccjs418e_fall2026/state_crime_intro.csv"
df = pd.read_csv(data_url)

In [ ]:
df.head(n=5)

In [ ]:
df.shape

The file has 20 state-year records: four states observed in five years. Because we are using the prepared file from the previous class, we can begin with the comparison rather than repeat the data-cleaning steps.


## Part 2: Conditions and Comparison Operators

A condition asks a yes-or-no question about every row. Pandas returns `True` when a row meets the condition and `False` when it does not.

| Operator | Meaning | Example question |
|---|---|---|
| `==` | Equal to | Is the state Maryland? |
| `!=` | Not equal to | Is the state not Maryland? |
| `>` | Greater than | Is the year after 2017? |
| `<` | Less than | Is the year before 2017? |
| `>=` | Greater than or equal to | Is the property-crime rate at least 2,000? |
| `<=` | Less than or equal to | Is the property-crime rate at most 1,500? |

A single equals sign, `=`, assigns a value. Two equals signs, `==`, compare values.


In [ ]:
is_maryland = df['state'] == 'Maryland'
is_maryland

The Boolean Series lines up with the DataFrame row by row. We place it inside `df[...]` to keep only the rows where the condition is `True`.


In [ ]:
maryland = df[is_maryland]
maryland

We can also put the condition directly inside the brackets.


In [ ]:
maryland = df[df['state'] == 'Maryland']
maryland

The other comparison operators work the same way. Each condition below selects a different set of state-year records.


In [ ]:
# != keeps rows that are not equal to the value
not_maryland = df[df['state'] != 'Maryland']
not_maryland[['state', 'year']].head(n=6)

In [ ]:
# > keeps years after 2017
after_2017 = df[df['year'] > 2017]
after_2017[['state', 'year']]

In [ ]:
# < keeps years before 2017
before_2017 = df[df['year'] < 2017]
before_2017[['state', 'year']]

In [ ]:
# >= includes 2,000 as well as values above it
at_least_2000 = df[df['property_all'] >= 2000]
at_least_2000[['state', 'year', 'property_all']]

In [ ]:
# <= includes 1,500 as well as values below it
at_most_1500 = df[df['property_all'] <= 1500]
at_most_1500[['state', 'year', 'property_all']]

## Part 3: Combine Conditions

A comparison often requires more than one condition. In pandas:

- `&` means **AND**: both conditions must be `True`.
- `|` means **OR**: at least one condition must be `True`.
- `~` means **NOT**: reverse `True` and `False`.

Put parentheses around each complete condition.


### AND: Require Both Conditions

To select Maryland records from 2018 onward, each row must meet both conditions.


In [ ]:
maryland_recent = df[
    (df['state'] == 'Maryland') &
    (df['year'] >= 2018)
]

maryland_recent

Use `&`, not the Python word `and`, when combining conditions across a pandas column.


### OR: Allow Either Condition

The next filter keeps the first or last year in the file.


In [ ]:
endpoints = df[
    (df['year'] == 2015) |
    (df['year'] == 2019)
]

endpoints[['state', 'year', 'property_all']]

Use `|`, not the Python word `or`, when combining conditions across a pandas column.


### NOT: Reverse a Condition

The condition `df['state'] == 'Maryland'` identifies Maryland rows. Adding `~` keeps every row that does not meet that condition.


In [ ]:
not_maryland = df[~(df['state'] == 'Maryland')]
not_maryland['state'].value_counts()

## Part 4: Select Several Values with `.isin()`

Writing a long chain of OR conditions can be difficult to read. The `.isin()` method checks whether each value appears in a list.


In [ ]:
comparison_states = ['Maryland', 'Virginia']

md_va = df[df['state'].isin(comparison_states)]
md_va[['state', 'year', 'property_all']]

We can combine `.isin()` with another condition. Here, we keep Maryland and Virginia records only when the year is 2019.


In [ ]:
md_va_2019 = df[
    (df['state'].isin(['Maryland', 'Virginia'])) &
    (df['year'] == 2019)
]

md_va_2019

## Part 5: Ratios, Percentages, and New Columns

Sometimes the measure needed for a comparison is not already in the table. We can calculate it from existing columns and save the result as a new column.

A **ratio** compares one quantity with another. The ratio below compares the motor-vehicle-theft rate with the burglary rate:

`motor-to-burglary ratio = motor-vehicle-theft rate / burglary rate`

A value of `0.50` would mean that the motor-vehicle-theft rate is half the burglary rate.


In [ ]:
df['motor_to_burglary_ratio'] = (
    df['property_motor'] / df['property_burglary']
)

df[['state', 'year', 'motor_to_burglary_ratio']].head(n=5)

A **percentage** describes a part relative to a whole. Motor vehicle theft is one part of total property crime, so we divide the motor-vehicle-theft rate by the total property-crime rate and multiply by 100:

`motor percentage = (motor-vehicle-theft rate / total property-crime rate) * 100`


In [ ]:
df['motor_pct'] = (
    df['property_motor'] / df['property_all']
) * 100

df[['state', 'year', 'property_all', 'property_motor', 'motor_pct']].head(n=5)

The new `motor_pct` column is not the percentage of residents who experienced motor vehicle theft. It is the percentage of the reported property-crime rate represented by motor vehicle theft.


## Part 6: Complete Analysis Workflow

We will now use the operations together to answer our question:

> In 2019, which state had the higher reported property-crime rate, Maryland or Virginia? Did that state also have a larger percentage of property crime represented by motor vehicle theft?

The comparison uses two state-year records: Maryland in 2019 and Virginia in 2019. The measures are the reported property-crime rate per 100,000 people and the calculated motor-vehicle-theft percentage.


### Step 1: Filter to the Records Needed


In [ ]:
analysis = df[
    (df['state'].isin(['Maryland', 'Virginia'])) &
    (df['year'] == 2019)
]

analysis

### Step 2: Select and Display the Measures


In [ ]:
results = analysis[
    ['state', 'year', 'property_all', 'property_motor', 'motor_pct']
].sort_values(by='property_all', ascending=False)

results.round(1)

### Step 3: Calculate the Size of Each Difference

Each filtered column contains two values, one for each state. The maximum minus the minimum gives the size of the difference.


In [ ]:
property_rate_difference = (
    analysis['property_all'].max() - analysis['property_all'].min()
)

property_rate_difference

In [ ]:
motor_pct_difference = (
    analysis['motor_pct'].max() - analysis['motor_pct'].min()
)

round(motor_pct_difference, 1)

### Step 4: Answer the Question

In 2019, Maryland had the higher reported property-crime rate: 1,950.2 compared with Virginia's 1,642.7 reported property crimes per 100,000 people. The difference was 307.5 per 100,000.

Maryland also had the larger motor-vehicle-theft percentage. Motor vehicle theft represented about 9.5% of Maryland's reported property-crime rate and 7.3% of Virginia's, a difference of about 2.2 percentage points.

The filtering identified the two records needed for the comparison. The calculated column let us compare the composition of property crime as well as the overall rate.


## Hands-On Exercise: Filter and Compare the State Crime Data

Complete each step using the methods from this notebook.


In [ ]:
# Step 1: Use comparison operators
# a. Keep records from years before 2018.
# b. Keep records with a property-crime rate greater than or equal to 2,200.
# c. Keep records where the state is not Delaware.


In [ ]:
# Step 2: Combine conditions
# a. Use AND to keep Maryland records from 2017 onward.
# b. Use OR to keep records from 2015 or 2019.
# c. Use NOT to keep every state except Virginia.


In [ ]:
# Step 3: Combine NOT with .isin()
# Keep records for every state except Maryland and Virginia.
# Use ~ with .isin() in one filter.
# Display state, year, and property_all for the resulting rows.


In [ ]:
# Step 4: Create two new columns
# a. Create burglary_to_motor_ratio by dividing property_burglary
#    by property_motor.
# b. Create burglary_pct by dividing property_burglary by
#    property_all and multiplying by 100.
# Display the first five rows of both new columns with state and year.


In [ ]:
# Step 5: Complete a comparison
# Compare Delaware and Pennsylvania in 2019.
# Which state had the higher reported property-crime rate?
# Did that state also have the larger burglary percentage?
#
# Filter to the two records, display the needed columns, and calculate:
# a. The difference in their property-crime rates.
# b. The difference in their burglary percentages.


## Reference: Filtering and Calculated Columns

```python
# Comparison operators
df['column'] == value
df['column'] != value
df['column'] > value
df['column'] < value
df['column'] >= value
df['column'] <= value

# One condition
filtered = df[df['column'] == value]

# Combined conditions
filtered = df[(condition_1) & (condition_2)]  # AND
filtered = df[(condition_1) | (condition_2)]  # OR
filtered = df[~(condition)]                   # NOT

# Several allowed values
filtered = df[df['column'].isin([value_1, value_2])]

# New columns
df['ratio'] = df['first_measure'] / df['second_measure']
df['percentage'] = (df['part'] / df['whole']) * 100
```

Common errors:

- Use `==` to compare values and `=` to assign a value or create a column.
- Put each condition in parentheses when using `&` or `|`.
- Use `&`, `|`, and `~` with pandas conditions rather than `and`, `or`, and `not`.
- Put the values used with `.isin()` inside a list.
- Check what belongs in the numerator and denominator before calculating a ratio or percentage.


## Wrap-Up: Key Takeaways

Today you learned to:

1. Use all six comparison operators to create Boolean conditions.
2. Filter a DataFrame to the records needed for a question.
3. Combine conditions with AND, OR, and NOT.
4. Select several possible values with `.isin()`.
5. Calculate ratios and percentages from existing measures.
6. Store a calculation in a new DataFrame column.
7. Move from a question to filtered records, calculated measures, results, and a supported answer.
